In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

import sys

# (Note: only run 1 time to change the directory!)
# Get the absolute path to the project root 
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Change the working directory 
os.chdir(project_root)

# Add to sys.path 
if project_root not in sys.path:
    sys.path.append(project_root)

print("Current Working Directory:", os.getcwd())

In [ ]:
import ipdb
from src.DataReaders import *
from src.address import *
from src.modelGen import *
from src.utils import cf_eval, cleanup_gpu
from models import CF_CausalCondLatentCF, CF_LatentCFpp, CF_PrototypeCF
from models import CF_CondLatentCF # Note: get the vanilla CF_CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [ ]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)


CF_METHODS = {  
    "LatentCFpp": CF_LatentCFpp,
    "Prototype": CF_PrototypeCF,
    "CausalCACTUS": CF_CausalCondLatentCF,
    "CACTUS": CF_CondLatentCF
}

In [ ]:
import json

# Load the config file for the experiment
with open("./configs/experiments/ABLATION_EXP1_CAUSAL.json") as f:
    EXP = json.load(f)

In [ ]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

file_path = "./configs/causal_graphs_rex/credit/"
# credit_rex5_constraints_min.dot from downsampled
with open(file_path + "constraints_min.dot", 'r') as file:
    graph_str1 = file.read()

# credit_rex6_constraints_mod.dot from downsampled
with open(file_path + "constraints_mod.dot", 'r') as file:
    graph_str2 = file.read()
    

In [ ]:
# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

## Graph 1 (Minimum priors)

In [ ]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AEmodel"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if "CausalCACTUS" in exp["name"]:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics1 = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics1 = df_metrics1.round(3)
print(df_metrics1)


In [ ]:
df_metrics1.to_csv("ablation-study1-caucal-cactus-graph1.csv")

In [ ]:
child_to_parents_dict

In [ ]:
# Pivoting the dataframe
df_metrics_pivot1 = df_metrics1.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot1.columns = df_metrics_pivot1.columns.swaplevel(0, 1)
df_metrics_pivot1 = df_metrics_pivot1[desired_order]

df_metrics_pivot1.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot1.columns = df_metrics_pivot1.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted1 = df_metrics_pivot1.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted1

## Graph 2 (Moderate priors)

In [ ]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AEmodel"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if "CausalCACTUS" in exp["name"]:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics2 = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics2 = df_metrics2.round(3)

In [ ]:
df_metrics2.to_csv("ablation-study1-caucal-cactus-graph2.csv")

In [ ]:
child_to_parents_dict

In [ ]:
# Pivoting the dataframe
df_metrics_pivot2 = df_metrics2.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot2.columns = df_metrics_pivot2.columns.swaplevel(0, 1)
df_metrics_pivot2 = df_metrics_pivot2[desired_order]

df_metrics_pivot2.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot2.columns = df_metrics_pivot2.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted2 = df_metrics_pivot2.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted2